# Embedding Evaluation — Notebook

End-to-end walkthrough using `emb_tight.json` (tight scenario) and `emb_sparse.json` (sparse scenario). Run all cells top-to-bottom; all figures are interactive (Plotly).

**Tight** — orthogonal class prototypes, tiny per-sample noise. Intra-class cosine ≈ 0.99, inter-class ≈ 0.00. All KPIs near-perfect.

**Sparse** — nearby class prototypes (~60° apart), large per-sample noise. Classes overlap heavily — gap ≈ 0.07, purity@5 ≈ 0.56.

In [16]:
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))

from pai.ag_emb.services.evaluate import run_evaluation
from pai.ag_emb.services.reporting import (
    plot_cosine_similarity,
    plot_knn_confusion,
    plot_lle,
    plot_tsne,
    print_result,
)

---

## Tight Scenario

### Load Data

In [17]:
payload_path = Path("emb_tight.json")
if not payload_path.exists():
    raise FileNotFoundError("emb_tight.json not found — start Jupyter from the examples/ directory")

with open(payload_path) as f:
    payload = json.load(f)

embeddings: dict[str, list[float]] = payload["embeddings"]
print(f"Loaded {len(embeddings)} embeddings  (dim={len(next(iter(embeddings.values())))})")

Loaded 23 embeddings  (dim=32)


### Run Evaluation

In [18]:
result = run_evaluation(
    image_embeddings=embeddings,
    k_values=[5, 10],
    dataset_root="images",
    sample_pairs=None,
)

print_result(result)

Confusion matrix: 100%|██████████| 8/8 [00:00<00:00, 55.06step/s]         

n_items      : 23
embedding_dim: 32
classes      : ['corn', 'soybean']
k_values     : [5, 10]

── global_metrics ──────────────────────────────────────────────────────
  pairwise cosine    : mean=0.5345  std=0.4725  (p05=-0.0237  p50=0.9187  p95=0.9907)
  centroid cosine    : mean=0.7448  std=0.2318  norm=0.7448
  intra/inter gap    : 0.9493  (intra=0.9548  inter=0.0055)

  KNN purity@5      : mean=1.0000  std=0.0000
  KNN purity@10     : mean=0.8783  std=0.1841
  nDCG@5           : mean=1.0000  std=0.0000
  nDCG@10          : mean=1.0000  std=0.0000
  MAP@5            : mean=0.4855  std=0.2301
  MAP@10           : mean=0.7681  std=0.1534

  effective_rank     : 1.22  (ratio=0.0380,  dim=32)

── per_class ───────────────────────────────────────────────────────────

  [corn]  n=16
    pairwise cosine  : mean=0.9563  std=0.0304
    centroid cosine  : mean=0.9793  norm=0.9793
    effective_rank   : 1.73  (ratio=0.0541)
    KNN purity@5     : mean=1.0000  std=0.0000  (p05=1.0000  p95=1.000

### Visualizations

All plots are interactive — hover for details, click legend entries to toggle classes, and drag to rotate 3D views.

#### KNN Confusion Matrix

Rows = true class, columns = neighbor class, values = fraction of k-NN neighbors belonging to each class. The diagonal equals mean KNN purity — higher is better.

In [19]:
plot_knn_confusion(result, output_path=None)

#### Pairwise Cosine Similarity

Full N×N cosine similarity matrix sorted by class. Within-class blocks sit on the diagonal — tighter, brighter blocks indicate a more discriminative embedding space.

In [20]:
plot_cosine_similarity(embeddings, result, output_path=None)

#### t-SNE — 2D

t-SNE preserves local neighborhood structure. Well-separated clusters indicate the model has learned class-discriminative features.

In [21]:
plot_tsne(embeddings, result, output_path=None, dimensions=2)

  t-SNE 2D — fitting 23 samples (perplexity=3, iter=1000)...


#### t-SNE — 3D

3D variant — drag to rotate, scroll to zoom.

In [22]:
plot_tsne(embeddings, result, output_path=None, dimensions=3)

  t-SNE 3D — fitting 23 samples (perplexity=3, iter=2000)...


#### LLE — 3D

Locally Linear Embedding preserves local geometry rather than global distances, complementing the t-SNE view. Drag to rotate.

In [23]:
plot_lle(embeddings, result, output_path=None)

  LLE 3D — fitting 23 samples (n_neighbors=3)...


---

## Sparse Scenario

Same image paths and class structure as the tight scenario, but class prototypes are close together (~60° apart) and per-sample noise is large. Compare KPIs and plots directly against the tight scenario above to see how embedding quality degrades.

### Load Data

In [24]:
sparse_path = Path("emb_sparse.json")
with open(sparse_path) as f:
    sparse_payload = json.load(f)

sparse_embeddings: dict[str, list[float]] = sparse_payload["embeddings"]
print(f"Loaded {len(sparse_embeddings)} sparse embeddings  (dim={len(next(iter(sparse_embeddings.values())))})")

Loaded 23 sparse embeddings  (dim=32)


### Run Evaluation

In [25]:
sparse_result = run_evaluation(
    image_embeddings=sparse_embeddings,
    k_values=[5, 10],
    dataset_root="images",
    sample_pairs=None,
)

print_result(sparse_result)

Confusion matrix: 100%|██████████| 8/8 [00:00<00:00, 54.65step/s]         

n_items      : 23
embedding_dim: 32
classes      : ['corn', 'soybean']
k_values     : [5, 10]

── global_metrics ──────────────────────────────────────────────────────
  pairwise cosine    : mean=0.2989  std=0.2826  (p05=-0.1287  p50=0.3227  p95=0.7003)
  centroid cosine    : mean=0.5739  std=0.1942  norm=0.5739
  intra/inter gap    : 0.4894  (intra=0.5155  inter=0.0262)

  KNN purity@5      : mean=0.9565  std=0.1313
  KNN purity@10     : mean=0.8609  std=0.2037
  nDCG@5           : mean=0.9692  std=0.0961
  nDCG@10          : mean=0.9774  std=0.0531
  MAP@5            : mean=0.4493  std=0.1992
  MAP@10           : mean=0.7293  std=0.1305

  effective_rank     : 6.25  (ratio=0.1952,  dim=32)

── per_class ───────────────────────────────────────────────────────────

  [corn]  n=16
    pairwise cosine  : mean=0.5312  std=0.1330
    centroid cosine  : mean=0.7487  norm=0.7487
    effective_rank   : 7.95  (ratio=0.2484)
    KNN purity@5     : mean=1.0000  std=0.0000  (p05=1.0000  p95=1.000

### Visualizations

All plots are interactive — hover for details, click legend entries to toggle classes, and drag to rotate 3D views.

#### KNN Confusion Matrix

Rows = true class, columns = neighbor class, values = fraction of k-NN neighbors belonging to each class. The diagonal equals mean KNN purity — higher is better.

In [26]:
plot_knn_confusion(sparse_result, output_path=None)

#### Pairwise Cosine Similarity

Full N×N cosine similarity matrix sorted by class. Within-class blocks sit on the diagonal — tighter, brighter blocks indicate a more discriminative embedding space.

In [27]:
plot_cosine_similarity(sparse_embeddings, sparse_result, output_path=None)

#### t-SNE — 2D

t-SNE preserves local neighborhood structure. Well-separated clusters indicate the model has learned class-discriminative features.

In [28]:
plot_tsne(sparse_embeddings, sparse_result, output_path=None, dimensions=2)

  t-SNE 2D — fitting 23 samples (perplexity=3, iter=1000)...


#### t-SNE — 3D

3D variant — drag to rotate, scroll to zoom.

In [29]:
plot_tsne(sparse_embeddings, sparse_result, output_path=None, dimensions=3)

  t-SNE 3D — fitting 23 samples (perplexity=3, iter=2000)...


#### LLE — 3D

Locally Linear Embedding preserves local geometry rather than global distances, complementing the t-SNE view. Drag to rotate.

In [30]:
plot_lle(sparse_embeddings, sparse_result, output_path=None)

  LLE 3D — fitting 23 samples (n_neighbors=3)...
